In [38]:
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import col, when
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [39]:
# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("ChurnPredictionPipeline") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

In [40]:
# 2. Load Data
df = spark.read.csv("churn_data.csv", header=True, inferSchema=True)

In [41]:
df.describe().show()

+-------+------------------+----------+------+------------------+-------+----------+------------------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+------------------+------------------+-----+
|summary|               _c0|customerID|gender|     SeniorCitizen|Partner|Dependents|            tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|    MonthlyCharges|      TotalCharges|Churn|
+-------+------------------+----------+------+------------------+-------+----------+------------------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+------------------+------------------+-----+
|  count|    

In [42]:
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



In [43]:
# 3. Data Cleaning
# Handle empty strings in TotalCharges and cast to Double
df = df.withColumn("TotalCharges", when(col("TotalCharges") == " ", "0.0").otherwise(col("TotalCharges")))
df = df.withColumn("TotalCharges", col("TotalCharges").cast(DoubleType()))
df = df.dropna()

In [44]:
# 4. Feature Engineering Configuration
categorical_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
                    'PaperlessBilling', 'PaymentMethod']
numeric_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

In [45]:
# Create StringIndexers for categorical columns
indexers = [StringIndexer(inputCol=c, outputCol=c+"_index", handleInvalid="keep") for c in categorical_cols]

# Index the target label (Churn)
label_indexer = StringIndexer(inputCol="Churn", outputCol="label")

# Assemble all features into a single vector
assembler_inputs = [c + "_index" for c in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="raw_features")

In [46]:
# Scale the features
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False)

# 5. Define Model
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100)

# 6. Build the Pipeline
stages = indexers + [label_indexer, assembler, scaler, rf]
pipeline = Pipeline(stages=stages)

# 7. Train/Test Split
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# 8. Train the Model
model = pipeline.fit(train_data)

# 9. Make Predictions
predictions = model.transform(test_data)

In [47]:
# 10. Evaluate Model
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)
print(f"Model AUC-ROC: {auc}")

# 11. Save the Model for Production
model.write().overwrite().save("churn_prediction_pipeline_model")

# Stop spark session
spark.stop()

Model AUC-ROC: 0.8296199848348189
